# 03 — State Space Models

SSMs provide an alternative to attention with linear complexity, grounded in control theory.

## HiPPO, S4, and Sequence Modelling

**State Space Models (SSMs)** are continuous-time dynamical systems:
$$\dot{x}(t) = Ax(t) + Bu(t)$$
$$y(t) = Cx(t) + Du(t)$$
where *u(t)* is the input, *x(t)* the hidden state, *y(t)* the output. For discrete sequences, this becomes:
$$x_k = \bar{A}x_{k-1} + \bar{B}u_k$$
$$y_k = Cx_k$$

**HiPPO** (Gu et al., 2020): designed a principled way to initialize *A* so the hidden state approximates the history of the input via orthogonal polynomial projection. This lets the SSM 'remember' long-range context in a compact hidden state.

**S4** (Structured State Space Sequences, Gu et al., 2021): makes SSMs practical by:
1. Restricting *A* to low-rank plus diagonal structure (Normal Plus Low-Rank)
2. Computing the convolution representation efficiently via FFT
3. Achieving O(n log n) training and O(1) per-step inference (recurrent mode)

**S5** simplifies S4 by using a diagonal *A* matrix and SISO (single-input single-output) state transitions, enabling parallel prefix scans during training.

In [ ]:
# S4/S5 style SSM implementation
import numpy as np
import torch
import torch.nn as nn

class S4Layer(nn.Module):
    """
    Simplified diagonal SSM (S5-style).
    A is diagonal; computed as convolution during training, recurrent during inference.
    """
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Log parametrisation for A (ensures stability: Re(A) < 0)
        self.A_log = nn.Parameter(torch.randn(d_state) * 0.5)
        self.B = nn.Parameter(torch.randn(d_state, d_model) * 0.02)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.02)
        self.D = nn.Parameter(torch.ones(d_model))

    def get_A(self):
        return -torch.exp(self.A_log)  # diagonal, negative for stability

    def forward_recurrent(self, u):
        """
        Recurrent mode: O(n) per sequence, O(d_state) memory.
        u: (batch, seq_len, d_model)
        """
        B, L, D = u.shape
        A = self.get_A()  # (d_state,)
        # Discretise: zero-order hold
        dt = torch.ones(D).to(u.device) * 0.1  # fixed step size
        A_bar = torch.exp(A.unsqueeze(0) * dt.unsqueeze(1))  # (D, d_state)
        B_bar = self.B * dt.unsqueeze(1) / (A.unsqueeze(0) + 1e-8)  # approx

        x = torch.zeros(B, D, self.d_state).to(u.device)
        outputs = []
        for i in range(L):
            # x: (batch, d_model, d_state); u_i: (batch, d_model)
            x = x * A_bar.unsqueeze(0) + u[:, i, :].unsqueeze(-1) * B_bar.unsqueeze(0)
            y_i = (x * self.C.T.unsqueeze(0)).sum(-1)  # (batch, d_model)
            outputs.append(y_i)

        y = torch.stack(outputs, dim=1)  # (batch, seq, d_model)
        return y + self.D * u

    def forward(self, u):
        return self.forward_recurrent(u)

# Test the SSM
torch.manual_seed(42)
ssm = S4Layer(d_model=32, d_state=8)
x = torch.randn(2, 64, 32)  # batch=2, seq=64, d=32
out = ssm(x)
print(f'S4 output shape: {out.shape}')

# Parameter count vs transformer
d_model = 128
seq_len = 512
ssm_params = S4Layer(d_model=d_model, d_state=16)
ssm_n = sum(p.numel() for p in ssm_params.parameters())

attn_params = (d_model * 3 * d_model) + d_model**2  # QKV + Out
print(f'SSM layer params: {ssm_n}')
print(f'Attention layer params: {attn_params}')
print(f'SSM is {attn_params/ssm_n:.1f}x lighter than attention')

In [ ]:
# SSM on a long-range dependency task
import matplotlib.pyplot as plt

# Copying task: model must remember first token and copy it T steps later
def make_copy_task(batch_size=32, seq_len=50, vocab_size=8):
    """Generate (input, target) pairs for copying task."""
    tokens = torch.randint(1, vocab_size, (batch_size, seq_len//2))
    padding = torch.zeros(batch_size, seq_len//4, dtype=torch.long)
    marker = torch.full((batch_size, 1), vocab_size, dtype=torch.long)
    # Input: tokens + padding + marker + zeros
    inputs = torch.cat([tokens, padding, marker, torch.zeros_like(padding)], dim=1)
    # Target: zeros + zeros + zeros + tokens
    targets = torch.cat([padding, padding, padding, tokens], dim=1)
    return inputs, targets

# Simple SSM-based model
class SSMModel(nn.Module):
    def __init__(self, vocab_size=9, d_model=32, d_state=16, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size + 1, d_model)
        self.ssm_layers = nn.ModuleList([S4Layer(d_model, d_state) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size + 1)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.ssm_layers:
            h = layer(h) + h  # residual
        return self.head(self.norm(h))

model = SSMModel()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for step in range(500):
    inputs, targets = make_copy_task()
    logits = model(inputs)
    loss = nn.CrossEntropyLoss(ignore_index=0)(logits.reshape(-1, 10), targets.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        losses.append(loss.item())
        print(f'Step {step}: loss={loss.item():.4f}')

print('SSM training complete')